In [5]:
import arcpy
from arcpy import metadata as md
import os

In [6]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# List of geodatabases containing items to export metadata for (individual HTML file output for each individual item in gdb)
gdbs = [
    r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Thesis_geodatabases\MASTER_THESIS.gdb",
    r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\Caribbean_shorelines.gdb",
    r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb",
    r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb",
    r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb",
    r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb"
]

# Root folder to export XML metadata files in
out_root = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\metadata\metadata_xml_files"

In [7]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun

In [8]:
# ── EXPORT ITEM METADATA IN XML FORMAT ───────────────────────────────────────────────────────────────

# Loop through each geodatbase ("gdb") in "gdbs" list (defined in path block)
for gdb in gdbs:
    # set arcpy workspace to current gdb in 'for' loop so 'List' functions read from it
    arcpy.env.workspace = gdb

    # extract name of gdb for naming subfolders (strip outer directories path and .gdb extension)
    gdb_name = os.path.splitext(os.path.basename(gdb))[0]
    # create subfolders for each gdb within root output directory (metadata_xml_files)
    out_dir = os.path.join(out_root, gdb_name)
    os.makedirs(out_dir, exist_ok=True)

    # collect all feature classes/rasters/tables/etc. in gdb
    items = []
    items += arcpy.ListFeatureClasses() or []
    items += arcpy.ListRasters() or []
    items += arcpy.ListTables() or []
    # also collect feature classes inside feature datasets (if applicable)
    for fds in arcpy.ListDatasets(feature_type="Feature") or []:
        for fc in arcpy.ListFeatureClasses(feature_dataset=fds) or []:
            items.append(os.path.join(fds, fc))

    # loop through each individual item in gdb
    for item in items:
        # build full path to item
        full_path = os.path.join(gdb, item)
        # flatten any dataset separators for clean filename
        safe_name = item.replace(os.sep, "_").replace("/", "_")
        # build output xml path from flattened name
        out_xml = os.path.join(out_dir, safe_name + ".xml")

        # wrap each export so one bad item does not stop the whole batch
        try:
            # load item metadata into a Metadata object
            m = md.Metadata(full_path)
            # saveAsXML writes complete ArcGIS-format metadata as-is (all content, no schema translation, no content-sensitive filtering)
            m.saveAsXML(out_xml)
            print(f"✅ Exported: {out_xml} \n")
        # print warning message if export fails
        except Exception as e:
            print(f"❌[FAILURE] {safe_name} in {gdb_name}.gdb: {e} \n")

✅ Exported: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\metadata\metadata_xml_files\MASTER_THESIS\boundary_thesis_p26920.xml 

✅ Exported: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\metadata\metadata_xml_files\Caribbean_shorelines\Caribbean_shorelines_all_p26920.xml 

✅ Exported: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\metadata\metadata_xml_files\Caribbean_shorelines\Caribbean_shorelines_all_p3857.xml 

✅ Exported: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\metadata\metadata_xml_files\Caribbean_shorelines\CARIBBEAN_SHORELINES_THESIS_p26920.xml 

✅ Exported: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\metadata\metadata_xml_files\GEBCO_Gridded2025_15arcsec\rdomain_GBC15a_bathy_entireEastCarib_g4326.xml 

✅ Exported: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\metadata\metadata_xml_files\GEBCO_Gridded2025_15arcs